# Regression Suite
# 0. 介绍

**研究背景**：Agent 的行为不只由大模型决定，还会随提示词、工具定义、上下文策略、执行环境和 Harness 代码一起变化。一次任务成功，只能说明当前组合在当前输入上成功；系统每次升级后，还需要重新检查过去已经修复的故障是否再次出现。

**现存问题**：工业界和生产系统中真实存在的错误基线，是只保留少量正常样例，升级后人工查看最终回复，或只比较一个平均成功率。这种做法没有保存故障发生时的输入、配置、执行轨迹和环境终态，也没有检查工具调用与状态变化；模型即使声称完成，实际产物仍可能缺字段。真实 API 还具有波动性，共享环境也可能残留状态，因此单次通过会漏掉旧错重现，单次失败也可能只是测试噪声，平均值则会掩盖某一类任务已经明显退化。

**解决方案**：本 Notebook 将实现一个极简的 Regression Suite，采用：`版本化 Golden Case + 隔离重放 + 分层确定性断言 + 统计回归门禁`。先把生产失败 trace 转成可版本化用例，固定任务、成功标准、模型与 Harness 配置；每次变更后在可重置环境中用真实 API 重放，并同时检查最终产物、工具调用和状态变化；再按任务类别比较通过率、Token、成本和延迟，只有关键用例全部通过且指标没有超过退化阈值才允许发布。后文会在同一任务和同一真实 API 决策下比较两种方式：错误基线只检查 Agent 的完成声明，因此放过残缺产物；改进版本依据环境终态执行确定性断言，准确发现旧故障并阻断发布，从而直观看到回归测试的核心不是“再跑一次”，而是把历史失败变成可重复、可判定、可执行的发布合同。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定任务与正确结果
回归测试必须先固定输入和正确结果，否则每次运行都没有统一的比较标准。下面使用一张常见的退款工单：任务要求模型填写金额、原因和审批状态，`expected_ticket` 则记录系统最终必须保存的完整内容。

In [2]:
# task 保存每次回放都相同的用户请求
# expected_ticket 保存判断任务是否成功的唯一标准
task = {
    "task_id": "refund_fields_001",
    "request": "请更新退款工单 RF-299：退款金额 299 元，退款原因是商品质量问题，审批状态改为等待财务复核。",
}
expected_ticket = {
    "ticket_id": "RF-299",
    "refund_amount": 299,
    "refund_reason": "商品质量问题",
    "approval_status": "等待财务复核",
}

print("任务编号：", task["task_id"])
print("用户请求：", task["request"])
print("正确结果：", expected_ticket)

任务编号： refund_fields_001
用户请求： 请更新退款工单 RF-299：退款金额 299 元，退款原因是商品质量问题，审批状态改为等待财务复核。
正确结果： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


输出把同一个任务和唯一正确结果放在一起。后续不论运行错误版本还是改进版本，输入和成功标准都不会改变；下一步只定义模型应当返回的结构化表单。

## 2.2 定义模型输出格式
只看自然语言回复很难判断字段是否齐全。下面用工具定义明确列出四个必填字段，让真实 API 返回可以直接交给程序处理的结构化参数。

In [3]:
# properties 说明每个字段的名称和数据类型
# required 说明四个字段缺少任何一个都不完整
ticket_tool = [{
    "type": "function",
    "function": {
        "name": "update_refund_ticket",
        "description": "把退款信息写入工单",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "refund_amount": {"type": "integer"},
                "refund_reason": {"type": "string"},
                "approval_status": {"type": "string"},
            },
            "required": ["ticket_id", "refund_amount", "refund_reason", "approval_status"],
        },
    },
}]

print("工具名称：", ticket_tool[0]["function"]["name"])
print("必填字段：", ticket_tool[0]["function"]["parameters"]["required"])

工具名称： update_refund_ticket
必填字段： ['ticket_id', 'refund_amount', 'refund_reason', 'approval_status']


输出显示模型必须提交同一张四字段表单。工具定义只约束模型如何表达决定，还没有执行写入；下一步准备一张空白工单，用它观察 Harness 实际保存了哪些字段。

## 2.3 准备可重置的工单环境
回归前后必须从相同状态开始，旧运行留下的数据不能混进新结果。下面定义一个极简环境函数，每次调用都会返回一张字段为空的全新工单。

In [4]:
def create_ticket_store():
    # 每次调用都新建字典，避免上一次运行污染下一次运行
    # 空值让后续输出可以直接显示哪些字段真正被写入
    return {
        "RF-299": {
            "ticket_id": "RF-299",
            "refund_amount": None,
            "refund_reason": None,
            "approval_status": None,
        }
    }

ticket_store = create_ticket_store()
print("初始工单：", ticket_store["RF-299"])

初始工单： {'ticket_id': 'RF-299', 'refund_amount': None, 'refund_reason': None, 'approval_status': None}


输出中的三个业务字段都是空值，说明环境没有带入历史结果。至此，任务、正确结果、工具格式和初始环境已经固定；下一章将把同一任务发送给真实 API，并查看模型返回的完整参数。

# 3. 获取并验证 API 响应
## 3.1 发送真实请求
现在把第 2 章固定的任务交给真实 API。请求明确要求模型填写 `update_refund_ticket` 工具表单，并同时记录输入上下文、模型、Token、延迟和停止原因，方便后续确认模型实际收到了什么、返回过程花了多少资源。

In [5]:
import time

# messages 是本次真实请求的完整输入上下文
# tool_choice 要求模型把决定填写进指定工具表单
messages = [
    {"role": "system", "content": "你负责填写退款工单。请只调用工具。"},
    {"role": "user", "content": task["request"]},
]
max_tokens = 256
started = time.perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=ticket_tool,
    tool_choice={"type": "function", "function": {"name": "update_refund_ticket"}},
    temperature=0,
    max_tokens=max_tokens,
)
latency_ms = round((time.perf_counter() - started) * 1000, 2)
usage = response.usage

print("输入上下文：", messages)
print("来源：task；Token 预算：", max_tokens)
print("Provider：", config["NANO_BACKEND"], "；模型：", model_name)
print("停止原因：", response.choices[0].finish_reason)
print("Token：", usage.prompt_tokens, "+", usage.completion_tokens)
print("延迟：", latency_ms, "ms；成本：未配置单价")

输入上下文： [{'role': 'system', 'content': '你负责填写退款工单。请只调用工具。'}, {'role': 'user', 'content': '请更新退款工单 RF-299：退款金额 299 元，退款原因是商品质量问题，审批状态改为等待财务复核。'}]
来源：task；Token 预算： 256
Provider： openai ；模型： LongCat-2.0
停止原因： tool_calls
Token： 232 + 204
延迟： 5124.3 ms；成本：未配置单价


输出证明本章已经向 `.env` 指定的真实模型发送了一次请求，并记录了输入来源、Token 预算、实际用量、延迟和停止原因。此时完整响应仍是 SDK 对象；下一步只读取其中的工具调用。

## 3.2 读取模型决定
工具参数在响应中是一段 JSON 文本。下面把它转换成普通 Python 字典，并打印调用编号、工具名称和四个参数；这些内容就是模型交给 Harness 的中间决定。

In [6]:
import json

# tool_call 保存模型选择的工具名称和调用编号
# json.loads 把参数文本转换成后续代码可直接使用的字典
tool_call = response.choices[0].message.tool_calls[0]
model_arguments = json.loads(tool_call.function.arguments)

print("调用编号：", tool_call.id)
print("工具名称：", tool_call.function.name)
print("模型参数：", model_arguments)

调用编号： call_74194f95ab9c474b8106c2d9
工具名称： update_refund_ticket
模型参数： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


输出显示模型已经提交结构化工具调用。调用编号把这次请求与后续工具结果关联起来，参数字典则会进入 Harness；下一步确认字段和值是否与固定答案完全相同。

## 3.3 对照正确结果
后续实验要把模型错误与 Harness 错误分开。下面直接比较模型参数和 `expected_ticket`；只有两边字段和值全部相同，后续出现的字段丢失才能明确归因于模型外层。

In [7]:
# 字典相等要求字段名称、字段数量和字段值全部相同
# True 表示模型决定完整，后续可以只改变 Harness 进行对照
response_matches_expected = model_arguments == expected_ticket

print("模型参数完整：", response_matches_expected)
print("正确结果：", expected_ticket)

模型参数完整： True
正确结果： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


`模型参数完整` 为 `True`，说明真实模型已经正确给出四个字段。模型决定现在成为后续两种 Harness 的共同输入；下一章将定义一个生产中常见的错误基线，让它在转交参数时静默丢掉合法字段。

# 4. 定义基线组件
## 4.1 定义过期字段映射
生产系统经常先只有编号和金额，后来才增加原因和状态。如果接口已经扩充，Harness 中的旧映射却没有同步，新字段就会在转交时被静默丢掉。下面故意保留这种常见错误：只复制最初的两个字段。

In [8]:
def outdated_field_mapping(arguments):
    # 旧代码仍然只认识最初的工单编号和退款金额
    # 合法的新字段不会报错，而是在这里被直接遗漏
    return {
        "ticket_id": arguments["ticket_id"],
        "refund_amount": arguments["refund_amount"],
    }

print("过期映射保留字段：ticket_id、refund_amount")

过期映射保留字段：ticket_id、refund_amount


输出明确显示旧映射只保留两个字段，退款原因和审批状态不会进入工具。单靠这段代码已经埋下数据缺失，但如果回归测试只看 Agent 的最后一句话，故障仍可能被放过；下一步定义这种错误判定。

## 4.2 定义只看回复的回归判定
另一种常见错误是把 Agent 的文字声明当成执行结果。下面的判定器只检查回复里是否出现“已更新完成”，完全不读取真实工单；因此文字听起来正确就会通过。

In [9]:
def claim_only_grader(final_message):
    # 这个错误判定只读取 Agent 的最终文字
    # 它不查看工具到底向环境写入了哪些字段
    return "已更新完成" in final_message

print("错误判定依据：只检查最终回复中的完成声明")

错误判定依据：只检查最终回复中的完成声明


输出说明基线判定与真实环境完全脱节。现在两个缺陷已经就绪：过期映射会丢字段，只看回复的判定又看不见字段缺失；下一章将实际运行这条链路，观察错误工单如何被误判为通过。

# 5. 展示基线故障
## 5.1 让旧映射截断模型参数
第 3 章已经证明模型给出了完整四字段。现在把这份正确参数交给过期映射，直接观察 Harness 在工具执行前实际保留了什么。

In [10]:
# model_arguments 是真实模型给出的完整决定
# baseline_arguments 是旧 Harness 实际转交给工具的参数
baseline_arguments = outdated_field_mapping(model_arguments)

print("调用编号：", tool_call.id)
print("模型原始参数：", model_arguments)
print("Harness 转交参数：", baseline_arguments)

调用编号： call_74194f95ab9c474b8106c2d9
模型原始参数： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}
Harness 转交参数： {'ticket_id': 'RF-299', 'refund_amount': 299}


输出显示同一个调用在进入 Harness 前有四个字段，转交给工具时只剩编号和金额。第一处分叉已经明确发生在字段映射，而不是模型决定；下一步把残缺参数写入全新的工单环境。

## 5.2 写入残缺工单
为了排除历史状态影响，下面先创建一张全新空白工单，再写入旧映射留下的两个字段。执行前后的字典会直接显示真实环境发生了什么变化。

In [11]:
# baseline_store 是本次基线运行独享的全新环境
# copy 保存写入前状态，便于和写入后结果直接比较
baseline_store = create_ticket_store()
before_ticket = baseline_store["RF-299"].copy()
baseline_store["RF-299"].update(baseline_arguments)
after_ticket = baseline_store["RF-299"]

print("写入前：", before_ticket)
print("写入后：", after_ticket)

写入前： {'ticket_id': 'RF-299', 'refund_amount': None, 'refund_reason': None, 'approval_status': None}
写入后： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': None, 'approval_status': None}


输出显示金额已经写入，但退款原因和审批状态仍为空。真实环境中的任务显然没有完成；下一步让错误回归判定只读取一条听起来成功的最终回复。

## 5.3 复现错误放行
Agent 最终回复“已更新完成”并不能证明工单正确。下面同时计算错误判定结果和真实产物结果；两者相反，就说明回归测试产生了假通过。

In [12]:
# baseline_gate_passed 只由 Agent 的文字声明决定
# artifact_is_correct 则比较环境工单与固定正确结果
final_message = "退款工单已更新完成"
baseline_gate_passed = claim_only_grader(final_message)
artifact_is_correct = after_ticket == expected_ticket
baseline_false_positive = baseline_gate_passed and not artifact_is_correct
baseline_result = {
    "grader": "claim_only",
    "gate_passed": baseline_gate_passed,
    "artifact_correct": artifact_is_correct,
    "false_positive": baseline_false_positive,
}

print("Agent 最终回复：", final_message)
print("错误判定通过：", baseline_gate_passed)
print("真实产物正确：", artifact_is_correct)
print("发生假通过：", baseline_false_positive)

Agent 最终回复： 退款工单已更新完成
错误判定通过： True
真实产物正确： False
发生假通过： True


错误判定通过为 `True`，真实产物正确为 `False`，因此发生假通过为 `True`。这复现了核心故障：模型决定正确，Harness 写入残缺，只看回复的回归测试却允许发布。下一章将定义以版本化用例和环境终态为核心的改进组件。

# 6. 定义改进组件
## 6.1 把失败固化为版本化用例
当前更可靠的做法，是把生产失败直接变成以后每次变更都必须重放的 Golden Case。下面保存固定编号、版本、原始请求、来源调用以及两层正确结果；这样用例不会只剩一句难以复现的故障描述。

In [13]:
# source_call_id 把回归用例连回产生故障的真实模型调用
# 两份期望值分别约束工具参数和最终环境产物
regression_case = {
    "case_id": "reg_refund_fields_001",
    "case_version": "1.0.0",
    "source_call_id": tool_call.id,
    "request": task["request"],
    "expected_tool_arguments": expected_ticket.copy(),
    "expected_artifact": expected_ticket.copy(),
}

print("用例编号：", regression_case["case_id"])
print("用例版本：", regression_case["case_version"])
print("来源调用：", regression_case["source_call_id"])
print("期望产物：", regression_case["expected_artifact"])

用例编号： reg_refund_fields_001
用例版本： 1.0.0
来源调用： call_74194f95ab9c474b8106c2d9
期望产物： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


输出表明历史失败已经成为有编号、有版本、有来源和明确终态的回归用例。以后 Harness 发生变化时可以重复使用同一合同；下一步修复导致字段丢失的映射。

## 6.2 定义完整字段透传
旧映射逐个挑选字段，因此接口增加字段后容易漏改。这里让 Harness 复制模型已经按工具格式提交的整份参数，避免在中间层再次维护一张过期字段表。

In [14]:
def complete_field_mapping(arguments):
    # 工具格式已经规定允许提交的四个字段
    # copy 保留全部参数，同时建立独立字典供工具写入
    return arguments.copy()

print("改进映射：复制工具调用中的全部字段")

改进映射：复制工具调用中的全部字段


输出说明改进映射不再维护第二份字段名单，模型提交的四个合法字段会一起到达工具。但参数完整不等于环境一定写对；下一步同时检查工具参数和最终工单。

## 6.3 定义两层确定性断言
可靠回归不能只检查一句回复。下面的评分器先比较工具实际收到的参数，再比较环境最终保存的工单；两层都等于 Golden Case 的期望值才算通过。

In [15]:
def deterministic_grader(tool_arguments, final_ticket, case):
    # 第一层定位参数是否在 Harness 中途丢失
    # 第二层确认工具执行后环境真的达到正确终态
    arguments_complete = tool_arguments == case["expected_tool_arguments"]
    artifact_correct = final_ticket == case["expected_artifact"]
    return {
        "arguments_complete": arguments_complete,
        "artifact_correct": artifact_correct,
        "passed": arguments_complete and artifact_correct,
    }

print("确定性断言：工具参数完整 + 环境终态正确")

确定性断言：工具参数完整 + 环境终态正确


输出给出了两层成功条件。任何一层失败都会让用例失败，并能直接看出问题发生在工具执行前还是环境写入后；下一步把这个结果变成清晰的发布决定。

## 6.4 定义发布门禁
回归结果只有影响发布流程才有约束力。这个 nano 套件只有一个关键用例，因此门禁规则很直接：用例通过才允许发布，用例失败就阻断发布。

In [16]:
def regression_gate(case_result):
    # passed 汇总了参数层和环境层的确定性断言
    # 返回值直接表示当前版本是否可以继续发布
    return case_result["passed"]

print("发布规则：关键回归用例通过，才允许发布")

发布规则：关键回归用例通过，才允许发布


输出说明回归结果已经连接到发布决定。至此，版本化用例、完整字段透传、两层确定性断言和发布门禁都已定义；下一章将从空白环境重放同一真实 API 决定，检查修复版本能否完整写入并通过门禁。

# 7. 展示修复结果
## 7.1 先拦截旧版本
改进后的回归套件首先要证明自己能发现历史故障。下面直接检查第 5 章留下的残缺参数和残缺工单；同一份错误产物必须让两层断言失败，并阻断发布。

In [17]:
# baseline_arguments 是旧映射转交给工具的残缺参数
# after_ticket 是旧工具执行后留在环境中的残缺工单
broken_case_result = deterministic_grader(
    baseline_arguments,
    after_ticket,
    regression_case,
)
broken_release_allowed = regression_gate(broken_case_result)

print("旧版本评分：", broken_case_result)
print("旧版本允许发布：", broken_release_allowed)

旧版本评分： {'arguments_complete': False, 'artifact_correct': False, 'passed': False}
旧版本允许发布： False


输出中参数完整、产物正确和总通过均为 `False`，发布也为 `False`。同一个旧版本不再因为一句“已完成”而通过，说明回归门禁已经能够抓住历史故障；下一步重放修复后的字段映射。

## 7.2 完整转交同一模型决定
为了只比较 Harness，本节不重新生成另一份模型决定。下面把第 3 章的同一份四字段参数交给完整映射，确认工具将收到全部内容。

In [18]:
# model_arguments 仍来自第 3 章的真实 API 调用
# fixed_arguments 是改进 Harness 实际交给工具的独立字典
fixed_arguments = complete_field_mapping(model_arguments)

print("调用编号：", tool_call.id)
print("模型原始参数：", model_arguments)
print("Harness 转交参数：", fixed_arguments)

调用编号： call_74194f95ab9c474b8106c2d9
模型原始参数： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}
Harness 转交参数： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


输出显示模型参数和 Harness 转交参数完全相同，退款原因与审批状态没有在中间层消失。下一步从空白工单开始执行写入，避免旧版本的环境状态影响结果。

## 7.3 写入全新工单
回归重放必须从一致的初始状态开始。下面重新创建空白环境，再把完整参数写入工单，并打印写入前后的状态变化。

In [19]:
# fixed_store 与基线环境相互独立，并从相同空白状态开始
# update 将四个完整字段一次写入当前工单
fixed_store = create_ticket_store()
fixed_before_ticket = fixed_store["RF-299"].copy()
fixed_store["RF-299"].update(fixed_arguments)
fixed_after_ticket = fixed_store["RF-299"]

print("写入前：", fixed_before_ticket)
print("写入后：", fixed_after_ticket)

写入前： {'ticket_id': 'RF-299', 'refund_amount': None, 'refund_reason': None, 'approval_status': None}
写入后： {'ticket_id': 'RF-299', 'refund_amount': 299, 'refund_reason': '商品质量问题', 'approval_status': '等待财务复核'}


输出显示金额、退款原因和审批状态都已写入，环境终态与 Golden Case 的目标一致。最后一步让确定性评分器和发布门禁正式给出结论。

## 7.4 验证并放行修复版本
下面使用与旧版本完全相同的用例和评分规则。只有工具参数与环境产物同时正确，门禁才允许修复版本发布。

In [20]:
# fixed_case_result 同时保存参数层和环境层的判定
# fixed_result 保留第 8 章消融对照需要的核心指标
fixed_case_result = deterministic_grader(
    fixed_arguments,
    fixed_after_ticket,
    regression_case,
)
fixed_release_allowed = regression_gate(fixed_case_result)
fixed_result = {
    "grader": "arguments_and_artifact",
    "gate_passed": fixed_release_allowed,
    "artifact_correct": fixed_case_result["artifact_correct"],
    "false_positive": fixed_release_allowed and not fixed_case_result["artifact_correct"],
}

print("修复版本评分：", fixed_case_result)
print("修复版本允许发布：", fixed_release_allowed)
print("发生假通过：", fixed_result["false_positive"])

修复版本评分： {'arguments_complete': True, 'artifact_correct': True, 'passed': True}
修复版本允许发布： True
发生假通过： False


参数完整、产物正确、总通过和允许发布均为 `True`，发生假通过为 `False`。修复没有更换模型、任务或正确答案，只改变字段透传与回归判定；这直接说明可靠性提升来自模型外层的 Harness。下一章将汇总有无 Regression Suite 的消融差异。

# 8. 汇总消融对照
## 8.1 对比三种组合
只比较错误基线和完整修复，会分不清提升来自回归套件还是字段修复。下面增加“有套件但仍用旧映射”这一组：它应当继续失败，却能正确阻断发布，从而单独证明 Regression Suite 的检测作用。

In [21]:
import pandas as pd

# 三组共享同一模型决定，只改变字段映射和回归判定
# 单任务成功率由环境产物是否等于 Golden Case 决定
baseline_success_rate = int(baseline_result["artifact_correct"]) * 100
blocked_success_rate = int(broken_case_result["artifact_correct"]) * 100
fixed_success_rate = int(fixed_case_result["artifact_correct"]) * 100
blocked_false_positive = broken_release_allowed and not broken_case_result["artifact_correct"]

ablation_rows = [
    {
        "方案": "无套件 + 旧映射",
        "转交字段数": len(baseline_arguments),
        "产物成功率": f"{baseline_success_rate}%",
        "允许发布": baseline_result["gate_passed"],
        "错误放行率": f"{int(baseline_result['false_positive']) * 100}%",
        "回归结论": "误放行",
    },
    {
        "方案": "有套件 + 旧映射",
        "转交字段数": len(baseline_arguments),
        "产物成功率": f"{blocked_success_rate}%",
        "允许发布": broken_release_allowed,
        "错误放行率": f"{int(blocked_false_positive) * 100}%",
        "回归结论": "正确阻断",
    },
    {
        "方案": "有套件 + 完整映射",
        "转交字段数": len(fixed_arguments),
        "产物成功率": f"{fixed_success_rate}%",
        "允许发布": fixed_release_allowed,
        "错误放行率": f"{int(fixed_result['false_positive']) * 100}%",
        "回归结论": "正确放行",
    },
]
ablation_table = pd.DataFrame(ablation_rows)

display(ablation_table)

,方案,转交字段数,产物成功率,允许发布,错误放行率,回归结论
0,无套件 + 旧映射,2,0%,True,100%,误放行
1,有套件 + 旧映射,2,0%,False,0%,正确阻断
2,有套件 + 完整映射,4,100%,True,0%,正确放行


表格显示，加入 Regression Suite 后，即使旧映射尚未修复，错误放行率也已从 `100%` 降为 `0%`，发布被正确阻断；再修复字段映射后，产物成功率从 `0%` 升为 `100%`，发布恢复放行。检测故障和修复故障是两个独立作用。

## 8.2 汇总真实 API 开销
三组 Harness 都复用第 3 章的同一次模型决定，因此模型、Token 和 API 延迟完全相同。下面汇总本次真实调用实际公开的数据；没有价格配置和缓存字段时保持未知，不猜测数值。

In [22]:
# 共享一次真实调用可以排除模型采样差异对消融结果的干扰
# 成本与缓存状态只记录当前 provider 实际能够提供的信息
runtime_summary = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "shared_call_id": tool_call.id,
    "api_calls": 1,
    "prompt_tokens": usage.prompt_tokens,
    "completion_tokens": usage.completion_tokens,
    "latency_ms": latency_ms,
    "cost_usd": None,
    "kv_cache_hit": "provider_not_exposed",
}

display(pd.DataFrame([runtime_summary]))

,provider,model,shared_call_id,api_calls,prompt_tokens,completion_tokens,latency_ms,cost_usd,kv_cache_hit
0,openai,LongCat-2.0,call_74194f95ab9c474b8106c2d9,1,232,204,5124.3,None,provider_not_exposed


表格记录了本次真实调用的 provider、模型、调用编号、Token 和延迟。三组只执行本地映射与判定，没有增加 API 调用；因此本实验能确认可靠性变化来自 Harness，但不宣称改进版本更快或更便宜。

## 8.3 收束因果结果
最后把完整实验方向压缩成一个布尔值：错误基线必须发生假通过，新门禁必须拦住旧版本，修复版本必须正确放行且不再假通过。

In [23]:
# 四个条件分别覆盖基线故障、回归拦截、修复成功和无假通过
# 只有整个因果方向一致，消融结论才为 True
ablation_matches_expected = (
    baseline_result["false_positive"]
    and not broken_release_allowed
    and fixed_release_allowed
    and not fixed_result["false_positive"]
)

print("消融方向符合预期：", ablation_matches_expected)

消融方向符合预期： True


`消融方向符合预期` 为 `True`。同一真实模型决定没有变化：错误基线把残缺产物误判为成功，Regression Suite 能阻断旧错，字段修复后再正确放行。这直接印证 binding-constraint thesis：决定 Agent 可靠性的关键，往往是模型外层能否保存失败、重放任务、检查真实终态并约束发布。

## 8.4 拓展

### nano 版省略了什么

nano 版只有三个固定用例和单次门禁，没有任务分层、历史失败自动入库、flaky test 隔离、基线版本、显著性检验、数据污染、跨模型矩阵和渐进发布。生产 Regression Suite 还应同时阻止质量、安全、成本与延迟退化，并保存触发失败的完整 trace 以便复现。

### 延伸阅读

1. 2026, [Anthropic, Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)：从真实失败构造任务集并用多类 grader 建立发布门禁。
2. 2026, [On Randomness in Agentic Evals](https://arxiv.org/abs/2602.07150)：避免把采样波动误判为回归或改进。
3. 2024, [Lessons from the Trenches on Reproducible Evaluation](https://arxiv.org/abs/2405.14782)：可复现评测的版本、数据与运行纪律。